In [3]:
!pip install langchain langgraph langsmith langchain-groq langchain_community

INFO: pip is looking at multiple versions of langchain-groq to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langgraph-prebuilt to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langgraph-prebuilt to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is still looking at multiple versions of langchain-community to determine which version is compatible with other requirements

In [17]:
groq_api_key = "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"

In [18]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_groq import ChatGroq

In [19]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-20b", groq_api_key=groq_api_key)


In [20]:
class State(TypedDict):
    message: Annotated[list, add_messages]
    improvements: list

In [21]:
def grammar_corrector(text: str) -> str:
    prompt = f"Correct the grammar in the following text without changing meaning:\n\n{text}"
    response = llm.invoke(prompt)
    return response.content

In [22]:
def sentence_rewriter(text: str) -> str:
    prompt = f"Rewrite the text to make it clearer and more natural:\n\n{text}"
    response = llm.invoke(prompt)
    return response.content


In [23]:
def tone_adjuster(text: str, tone: str = "formal") -> str:
    prompt = f"Rewrite the text in a {tone} tone:\n\n{text}"
    response = llm.invoke(prompt)
    return response.content

In [24]:
def decide_tools(state: State):
    user_input = state["message"][-1].content

    improvements = []
    # Always do grammar correction and rewriting
    grammar_fixed = grammar_corrector(user_input)
    rewritten = sentence_rewriter(grammar_fixed)
    improvements.append(("Grammar Fix", grammar_fixed))
    improvements.append(("Rewritten", rewritten))

    # If user explicitly requests tone adjustment
    if "tone:" in user_input.lower():
        tone = user_input.lower().split("tone:")[-1].strip()
        toned = tone_adjuster(rewritten, tone)
        improvements.append((f"Tone Adjusted ({tone})", toned))

    return {"improvements": improvements}

In [25]:
def generate_response(state: State):
    response = "Here are the improvements I made:\n"
    for label, text in state["improvements"]:
        response += f"\n🔹 {label}:\n{text}\n"
    return {"message": [{"role": "assistant", "content": response}]}

In [26]:
workflow = StateGraph(State)
workflow.add_node("decide_tools", decide_tools)
workflow.add_node("generate_response", generate_response)

workflow.add_edge(START, "decide_tools")
workflow.add_edge("decide_tools", "generate_response")
workflow.add_edge("generate_response", END)

app = workflow.compile()

In [27]:
print("📝 AI Writing Assistant (type 'exit' to quit)\n")

📝 AI Writing Assistant (type 'exit' to quit)



In [28]:
while True:
    user_text = input("You: ")
    if user_text.lower() in ["exit", "quit"]:
        print("👋 Goodbye!")
        break

    inputs = {"message": [{"role": "user", "content": user_text}]}

    for output in app.stream(inputs):
        for key, value in output.items():
            if key == "generate_response":
                print("\nAssistant:\n")
                print(value["message"][-1]["content"])
                print("-" * 50)

You: what are you name

Assistant:

Here are the improvements I made:

🔹 Grammar Fix:
What is your name?

🔹 Rewritten:
What’s your name?

--------------------------------------------------
You: exit
👋 Goodbye!


In [29]:
!pip install gradio

In [37]:
import gradio as gr

# Dummy function; replace with your actual streaming logic
def respond(user_message, chat_history):
    chat_history = chat_history or []
    response_text = ""

    inputs = {"message": [{"role": "user", "content": user_message}]}

    for output in app.stream(inputs):
        if "generate_response" in output:
            response_text += output["generate_response"]["message"][-1]["content"]

    # For type='messages', each entry is a dict with 'role' and 'content'
    chat_history.append({"role": "user", "content": user_message})
    chat_history.append({"role": "assistant", "content": response_text})
    return chat_history

with gr.Blocks(css="""
    .chatbox {border-radius: 15px; border: 2px solid #4A90E2; padding: 10px; background-color: #f0f4f8;}
    .textbox_style {border-radius: 12px; border: 1px solid #4A90E2; padding: 10px; width: 100%;}
    .gr-button {border-radius: 12px; background-color: #4A90E2; color: white;}
""") as demo:
    gr.Markdown("## 🤖 AI Assistant", elem_id="title", visible=True)

    with gr.Row():
        chatbot = gr.Chatbot(elem_classes="chatbox", type="messages")

    with gr.Row():
        msg = gr.Textbox(placeholder="Type your message here...", elem_classes="textbox_style")
        clear = gr.Button("Clear")

    msg.submit(respond, [msg, chatbot], chatbot)
    clear.click(lambda: [], None, chatbot, queue=False)

demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6c269d735b825dd3a1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
